# 🏥 HakimBot — Medical Assistant AI

**Pipeline:**
```
Understand → Decision Gate → Planner → Retrieval (parallel/sequential)
          → Verification → Smart Retry → Smart Merge → Answer
```

**General LLM roles:** 💬 chit-chat &nbsp;|&nbsp; ❓ clarify &nbsp;|&nbsp; 🛟 fallback

Run all cells top to bottom, then `launch_gradio(share=True)`.


---
## ⚙️ Section 1 — Install

In [ ]:
# =========================================
# 🔧 Environment Setup (All Dependencies)
# =========================================

!pip install --upgrade pip

# Core AI / LLM / LangChain
!pip install groq
!pip install langgraph
!pip install langchain-core
!pip install langchain-community
!pip install langchain-experimental

# Embeddings & NLP
!pip install sentence-transformers
!pip install transformers
!pip install torch

# Vector DB
!pip install chromadb

# Data Processing
!pip install pandas
!pip install "numpy>=2.0"

# Document Processing
!pip install pymupdf
!pip install pdfplumber
!pip install python-docx

# OCR
!apt-get install -y tesseract-ocr
!pip install pytesseract
!pip install pillow

# Web Scraping
!pip install requests
!pip install playwright
!playwright install --with-deps

# OpenCV
!pip install opencv-python-headless

# Utilities
!pip install tqdm

print("🏥 HakimBot environment is ready!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 63 not upgraded.
Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis

---
## 📦 Section 2 — Imports & Config

In [ ]:
from __future__ import annotations

import asyncio
import glob
import hashlib
import io
import json
import os
import re
import shutil
import sqlite3
from collections import Counter
from datetime import datetime, timedelta
from typing import Annotated, Literal, Sequence, TypedDict

import chromadb
import fitz
import pandas as pd
import pdfplumber
import pytesseract
import requests
from docx import Document as DocxDoc
import urllib.request
from groq import Groq
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from PIL import Image
from playwright.async_api import async_playwright
from sentence_transformers import SentenceTransformer

In [ ]:
# -------------------------------------
# Environment & clients
# -------------------------------------

GROQ_API_KEY = "your Key"
GROQ_MODEL   = "llama-3.3-70b-versatile"
EMBED_MODEL  = "intfloat/multilingual-e5-small"

DB_PATH = os.environ.get("CLINICIQ_DB_PATH", "cliniciq.db")

PDF_URLS: dict[str, str] = {}
WEBSITES: dict[str, str] = {}

groq_client: Groq | None = None
embedder: SentenceTransformer | None = None
collection = None  # Chroma collection

memory: list[dict] = []
processed_files: set[str] = set()


def _get_groq() -> Groq:
    global groq_client
    if groq_client is None:
        if not GROQ_API_KEY:
            raise RuntimeError("Set GROQ_API_KEY environment variable.")
        groq_client = Groq(api_key=GROQ_API_KEY)
    return groq_client

---
## 🗄️ Section 3 — SQL Layer (Medical Database)

In [ ]:
# =============================================================================
# SQL layer — Medical clinic database + RBAC
# Roles: 1=Receptionist, 2=Doctor, 3=Admin
# =============================================================================


def init_db() -> None:
    """Create schema, views, and seed sample data for ClinicIQ demo DB."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    c.executescript(
        """
    -- Clinics / Branches
    CREATE TABLE IF NOT EXISTS Clinics (
        ClinicId    INTEGER PRIMARY KEY AUTOINCREMENT,
        ClinicName  TEXT NOT NULL,
        Specialty   TEXT,
        Address     TEXT,
        Phone       TEXT
    );

    -- Doctors
    CREATE TABLE IF NOT EXISTS Doctors (
        DoctorId    INTEGER PRIMARY KEY AUTOINCREMENT,
        ClinicId    INTEGER,
        FullName    TEXT NOT NULL,
        Specialty   TEXT,
        Phone       TEXT,
        Email       TEXT,
        Role        INTEGER DEFAULT 2
    );

    -- Staff (Receptionists / Admins)
    CREATE TABLE IF NOT EXISTS Staff (
        StaffId     INTEGER PRIMARY KEY AUTOINCREMENT,
        ClinicId    INTEGER,
        FullName    TEXT NOT NULL,
        Position    TEXT,
        Phone       TEXT,
        Email       TEXT,
        Role        INTEGER DEFAULT 1
    );

    -- Patients
    CREATE TABLE IF NOT EXISTS Patients (
        PatientId       INTEGER PRIMARY KEY AUTOINCREMENT,
        FullName        TEXT NOT NULL,
        DateOfBirth     TEXT,
        Gender          TEXT,
        Phone           TEXT,
        NationalId      TEXT UNIQUE,
        BloodType       TEXT,
        KnownAllergies  TEXT,
        ChronicDiseases TEXT,
        InsuranceNo     TEXT,
        RegisteredAt    TEXT DEFAULT CURRENT_TIMESTAMP
    );

    -- Appointments
    CREATE TABLE IF NOT EXISTS Appointments (
        AppointmentId   INTEGER PRIMARY KEY AUTOINCREMENT,
        PatientId       INTEGER,
        DoctorId        INTEGER,
        ClinicId        INTEGER,
        AppointmentDate TEXT,
        AppointmentTime TEXT,
        Reason          TEXT,
        Status          INTEGER DEFAULT 1,
        Notes           TEXT
    );
    -- Status: 1=Scheduled, 2=Completed, 3=Cancelled, 4=NoShow

    -- Medical Records (Visits)
    CREATE TABLE IF NOT EXISTS MedicalRecords (
        RecordId        INTEGER PRIMARY KEY AUTOINCREMENT,
        PatientId       INTEGER,
        DoctorId        INTEGER,
        VisitDate       TEXT,
        Diagnosis       TEXT,
        Symptoms        TEXT,
        TreatmentPlan   TEXT,
        Prescriptions   TEXT,
        LabTests        TEXT,
        FollowUpDate    TEXT,
        Notes           TEXT
    );

    -- Prescriptions
    CREATE TABLE IF NOT EXISTS Prescriptions (
        PrescriptionId  INTEGER PRIMARY KEY AUTOINCREMENT,
        RecordId        INTEGER,
        PatientId       INTEGER,
        DoctorId        INTEGER,
        MedicationName  TEXT NOT NULL,
        Dosage          TEXT,
        Frequency       TEXT,
        DurationDays    INTEGER,
        IssuedAt        TEXT DEFAULT CURRENT_TIMESTAMP
    );

    -- Lab Results
    CREATE TABLE IF NOT EXISTS LabResults (
        LabId           INTEGER PRIMARY KEY AUTOINCREMENT,
        PatientId       INTEGER,
        DoctorId        INTEGER,
        RecordId        INTEGER,
        TestName        TEXT NOT NULL,
        Result          TEXT,
        ReferenceRange  TEXT,
        Status          INTEGER DEFAULT 1,
        TestedAt        TEXT DEFAULT CURRENT_TIMESTAMP
    );
    -- Status: 1=Pending, 2=Ready, 3=Reviewed

    -- Invoices / Billing
    CREATE TABLE IF NOT EXISTS Invoices (
        InvoiceId       INTEGER PRIMARY KEY AUTOINCREMENT,
        PatientId       INTEGER,
        AppointmentId   INTEGER,
        Amount          REAL,
        PaidAmount      REAL DEFAULT 0,
        PaymentStatus   INTEGER DEFAULT 1,
        IssuedAt        TEXT DEFAULT CURRENT_TIMESTAMP
    );
    -- PaymentStatus: 1=Pending, 2=Paid, 3=PartiallyPaid, 4=Waived
    """
    )

    # ── Views ──────────────────────────────────────────────────────────────────
    c.executescript(
        """
    DROP VIEW IF EXISTS AppointmentFull;
    CREATE VIEW AppointmentFull AS
    SELECT
        a.AppointmentId,
        p.FullName        AS PatientName,
        p.Phone           AS PatientPhone,
        p.BloodType,
        p.KnownAllergies,
        p.ChronicDiseases,
        d.FullName        AS DoctorName,
        d.Specialty       AS DoctorSpecialty,
        cl.ClinicName,
        a.AppointmentDate,
        a.AppointmentTime,
        a.Reason,
        CASE a.Status
            WHEN 1 THEN 'Scheduled'
            WHEN 2 THEN 'Completed'
            WHEN 3 THEN 'Cancelled'
            WHEN 4 THEN 'NoShow'
            ELSE 'Unknown' END AS Status,
        a.Notes
    FROM Appointments a
    LEFT JOIN Patients p  ON a.PatientId = p.PatientId
    LEFT JOIN Doctors  d  ON a.DoctorId  = d.DoctorId
    LEFT JOIN Clinics  cl ON a.ClinicId  = cl.ClinicId;

    DROP VIEW IF EXISTS MedicalRecordFull;
    CREATE VIEW MedicalRecordFull AS
    SELECT
        mr.RecordId,
        p.FullName       AS PatientName,
        p.BloodType,
        p.KnownAllergies,
        p.ChronicDiseases,
        d.FullName       AS DoctorName,
        d.Specialty      AS DoctorSpecialty,
        mr.VisitDate,
        mr.Diagnosis,
        mr.Symptoms,
        mr.TreatmentPlan,
        mr.Prescriptions,
        mr.LabTests,
        mr.FollowUpDate,
        mr.Notes
    FROM MedicalRecords mr
    LEFT JOIN Patients p ON mr.PatientId = p.PatientId
    LEFT JOIN Doctors  d ON mr.DoctorId  = d.DoctorId;

    DROP VIEW IF EXISTS PatientSummary;
    CREATE VIEW PatientSummary AS
    SELECT
        p.PatientId,
        p.FullName,
        p.DateOfBirth,
        p.Gender,
        p.Phone,
        p.NationalId,
        p.BloodType,
        p.KnownAllergies,
        p.ChronicDiseases,
        p.InsuranceNo,
        COUNT(DISTINCT a.AppointmentId) AS TotalAppointments,
        COUNT(DISTINCT mr.RecordId)     AS TotalVisits,
        MAX(a.AppointmentDate)          AS LastAppointmentDate
    FROM Patients p
    LEFT JOIN Appointments   a  ON p.PatientId = a.PatientId
    LEFT JOIN MedicalRecords mr ON p.PatientId = mr.PatientId
    GROUP BY p.PatientId;

    DROP VIEW IF EXISTS LabResultsFull;
    CREATE VIEW LabResultsFull AS
    SELECT
        l.LabId,
        p.FullName   AS PatientName,
        d.FullName   AS DoctorName,
        l.TestName,
        l.Result,
        l.ReferenceRange,
        CASE l.Status
            WHEN 1 THEN 'Pending'
            WHEN 2 THEN 'Ready'
            WHEN 3 THEN 'Reviewed'
            ELSE 'Unknown' END AS LabStatus,
        l.TestedAt
    FROM LabResults l
    LEFT JOIN Patients p ON l.PatientId = p.PatientId
    LEFT JOIN Doctors  d ON l.DoctorId  = d.DoctorId;

    DROP VIEW IF EXISTS InvoiceView;
    CREATE VIEW InvoiceView AS
    SELECT
        i.InvoiceId,
        p.FullName    AS PatientName,
        i.Amount,
        i.PaidAmount,
        (i.Amount - i.PaidAmount) AS BalanceDue,
        CASE i.PaymentStatus
            WHEN 1 THEN 'Pending'
            WHEN 2 THEN 'Paid'
            WHEN 3 THEN 'PartiallyPaid'
            WHEN 4 THEN 'Waived'
            ELSE 'Unknown' END AS PaymentStatus,
        i.IssuedAt
    FROM Invoices i
    LEFT JOIN Patients p ON i.PatientId = p.PatientId;
    """
    )

    # ── Seed Data ──────────────────────────────────────────────────────────────
    today = datetime.now()

    # Clinics
    for cl in [
        (1, 'Cairo Medical Center',   'General Practice', 'Nasr City, Cairo',     '02-22345678'),
        (2, 'Al-Shifa Cardiology',    'Cardiology',       'Heliopolis, Cairo',    '02-24456789'),
        (3, 'Nile Pediatric Clinic',  'Pediatrics',       'Dokki, Giza',          '02-37891234'),
    ]:
        c.execute('INSERT OR IGNORE INTO Clinics VALUES(?,?,?,?,?)', cl)

    # Doctors
    for d in [
        (1, 1, 'Dr. Ahmed Samir',     'General Practice', '0100-1234567', 'ahmed.samir@cmc.eg',    3),
        (2, 2, 'Dr. Mona Hassan',     'Cardiology',       '0101-2345678', 'mona.hassan@shifa.eg',  2),
        (3, 3, 'Dr. Khaled Nour',     'Pediatrics',       '0102-3456789', 'khaled.nour@nile.eg',   2),
        (4, 1, 'Dr. Sara El-Gohary',  'Internal Medicine','0103-4567890', 'sara.gohary@cmc.eg',    2),
        (5, 2, 'Dr. Tarek Mansour',   'Cardiology',       '0104-5678901', 'tarek.mansour@shifa.eg',2),
    ]:
        c.execute('INSERT OR IGNORE INTO Doctors VALUES(?,?,?,?,?,?,?)', d)

    # Staff
    for s in [
        (1, 1, 'Rania Fouad',   'Receptionist', '0105-6789012', 'rania@cmc.eg',   1),
        (2, 2, 'Omar Salah',    'Receptionist', '0106-7890123', 'omar@shifa.eg',  1),
        (3, 1, 'Hana Ibrahim',  'Admin',        '0107-8901234', 'hana@cmc.eg',    3),
    ]:
        c.execute('INSERT OR IGNORE INTO Staff VALUES(?,?,?,?,?,?,?)', s)

    # Patients
    for p in [
        (1,  'Mohamed Ali Hassan',   '1985-03-15', 'Male',   '0111-1111111', '28503152345', 'A+',  'Penicillin',        'Hypertension, Diabetes Type 2', 'INS-001'),
        (2,  'Fatma Mahmoud',        '1992-07-22', 'Female', '0122-2222222', '29207224567', 'B+',  None,                'Asthma',                        'INS-002'),
        (3,  'Youssef Kamal',        '1978-11-08', 'Male',   '0133-3333333', '27811083456', 'O+',  'Aspirin, Ibuprofen','Coronary Artery Disease',       'INS-003'),
        (4,  'Nadia Saber',          '2015-05-30', 'Female', '0144-4444444', '31505302341', 'AB-', None,                None,                             'INS-004'),
        (5,  'Hossam Fathy',         '1967-09-12', 'Male',   '0155-5555555', '26709124678', 'A-',  'Sulfa drugs',       'Hypertension, Hypothyroidism',  'INS-005'),
        (6,  'Amira Gamal',          '1999-01-18', 'Female', '0166-6666666', '29901185678', 'O-',  None,                None,                             'INS-006'),
        (7,  'Ibrahim Youssef',      '1955-06-25', 'Male',   '0177-7777777', '25506256789', 'B-',  'Codeine',           'COPD, Diabetes Type 2',         'INS-007'),
        (8,  'Salma Hassan',         '2010-12-03', 'Female', '0188-8888888', '31012035678', 'A+',  None,                'Asthma',                        'INS-008'),
        (9,  'Walid Mostafa',        '1980-04-19', 'Male',   '0199-9999999', '28004197890', 'AB+', 'Latex',             None,                             'INS-009'),
        (10, 'Doha El-Sayed',        '1995-08-07', 'Female', '0100-0000001', '29508071234', 'B+',  None,                'Migraine',                      'INS-010'),
    ]:
        c.execute('INSERT OR IGNORE INTO Patients VALUES(?,?,?,?,?,?,?,?,?,?,CURRENT_TIMESTAMP)', p)

    # Appointments
    appts = [
        (1, 1,  1, 1, (today+timedelta(days=1)).strftime('%Y-%m-%d'),  '09:00', 'Follow-up on blood pressure',          1, None),
        (2, 2,  2, 2, (today+timedelta(days=2)).strftime('%Y-%m-%d'),  '10:30', 'Chest pain evaluation',                1, None),
        (3, 3,  5, 2, (today+timedelta(days=1)).strftime('%Y-%m-%d'),  '11:00', 'Routine cardiac checkup',              1, None),
        (4, 4,  3, 3, (today+timedelta(days=3)).strftime('%Y-%m-%d'),  '08:30', 'Child fever and cough',                1, None),
        (5, 5,  4, 1, (today-timedelta(days=7)).strftime('%Y-%m-%d'),  '09:30', 'Blood pressure management',            2, 'BP stabilized, medication adjusted'),
        (6, 6,  4, 1, (today-timedelta(days=5)).strftime('%Y-%m-%d'),  '14:00', 'General checkup',                      2, 'All normal'),
        (7, 7,  5, 2, (today-timedelta(days=3)).strftime('%Y-%m-%d'),  '10:00', 'Breathing difficulties',               2, 'COPD management plan updated'),
        (8, 8,  3, 3, (today-timedelta(days=2)).strftime('%Y-%m-%d'),  '11:30', 'Asthma inhaler review',                2, 'New inhaler prescribed'),
        (9, 9,  1, 1, (today+timedelta(days=5)).strftime('%Y-%m-%d'),  '15:00', 'Annual checkup',                       1, None),
        (10,10, 4, 1, (today-timedelta(days=1)).strftime('%Y-%m-%d'),  '09:00', 'Severe headache',                      3, 'Patient cancelled'),
    ]
    for a in appts:
        c.execute('INSERT OR IGNORE INTO Appointments VALUES(?,?,?,?,?,?,?,?,?)', a)

    # Medical Records
    records = [
        (1, 1, 1, (today-timedelta(days=14)).strftime('%Y-%m-%d'), 'Hypertension Stage 1',
         'Headache, dizziness, elevated BP (150/95)',
         'Lifestyle modification + medication',
         'Amlodipine 5mg daily, Lisinopril 10mg daily',
         'CBC, Lipid Panel, Creatinine',
         (today+timedelta(days=30)).strftime('%Y-%m-%d'), 'Monitor BP weekly'),

        (2, 3, 5, (today-timedelta(days=10)).strftime('%Y-%m-%d'), 'Stable Angina',
         'Chest tightness on exertion, shortness of breath',
         'Medication adjustment, reduce physical stress',
         'Nitroglycerin PRN, Atorvastatin 40mg, Aspirin 81mg',
         'ECG, Stress Test, Echocardiogram',
         (today+timedelta(days=14)).strftime('%Y-%m-%d'), 'Avoid heavy exertion'),

        (3, 5, 4, (today-timedelta(days=7)).strftime('%Y-%m-%d'), 'Hypertension + Hypothyroidism',
         'Fatigue, cold intolerance, BP 160/100',
         'Thyroid hormone replacement + antihypertensive',
         'Levothyroxine 50mcg, Amlodipine 10mg',
         'TSH, Free T4, BMP',
         (today+timedelta(days=21)).strftime('%Y-%m-%d'), 'Recheck TSH in 6 weeks'),

        (4, 7, 5, (today-timedelta(days=3)).strftime('%Y-%m-%d'), 'COPD Exacerbation',
         'Increased dyspnea, productive cough, wheezing',
         'Bronchodilators, antibiotics, pulmonary rehab',
         'Salbutamol inhaler, Azithromycin 500mg x5 days, Prednisolone',
         'Spirometry, CXR, Sputum Culture',
         (today+timedelta(days=7)).strftime('%Y-%m-%d'), 'Strict smoking cessation counseling'),

        (5, 8, 3, (today-timedelta(days=2)).strftime('%Y-%m-%d'), 'Mild Persistent Asthma',
         'Nocturnal cough, exercise-induced wheeze',
         'Step-up therapy with ICS/LABA combination',
         'Fluticasone/Salmeterol inhaler, Salbutamol rescue',
         'Peak Flow Measurement, Spirometry',
         (today+timedelta(days=30)).strftime('%Y-%m-%d'), 'Asthma action plan given to parents'),
    ]
    for r in records:
        c.execute('INSERT OR IGNORE INTO MedicalRecords VALUES(?,?,?,?,?,?,?,?,?,?,?)', r)

    # Prescriptions
    rxs = [
        (1,  1, 1, 1, 'Amlodipine',         '5mg',   'Once daily',   30, (today-timedelta(days=14)).strftime('%Y-%m-%d')),
        (2,  1, 1, 1, 'Lisinopril',          '10mg',  'Once daily',   30, (today-timedelta(days=14)).strftime('%Y-%m-%d')),
        (3,  2, 3, 5, 'Nitroglycerin',       'PRN',   'As needed',    30, (today-timedelta(days=10)).strftime('%Y-%m-%d')),
        (4,  2, 3, 5, 'Atorvastatin',        '40mg',  'Once nightly', 30, (today-timedelta(days=10)).strftime('%Y-%m-%d')),
        (5,  2, 3, 5, 'Aspirin',             '81mg',  'Once daily',   30, (today-timedelta(days=10)).strftime('%Y-%m-%d')),
        (6,  3, 5, 4, 'Levothyroxine',       '50mcg', 'Once morning', 90, (today-timedelta(days=7)).strftime('%Y-%m-%d')),
        (7,  3, 5, 4, 'Amlodipine',          '10mg',  'Once daily',   30, (today-timedelta(days=7)).strftime('%Y-%m-%d')),
        (8,  4, 7, 5, 'Azithromycin',        '500mg', 'Once daily x5',5,  (today-timedelta(days=3)).strftime('%Y-%m-%d')),
        (9,  4, 7, 5, 'Prednisolone',        '40mg',  'Once daily x7',7,  (today-timedelta(days=3)).strftime('%Y-%m-%d')),
        (10, 5, 8, 3, 'Fluticasone/Salm.',   '1 puff','Twice daily',  30, (today-timedelta(days=2)).strftime('%Y-%m-%d')),
    ]
    for rx in rxs:
        c.execute('INSERT OR IGNORE INTO Prescriptions VALUES(?,?,?,?,?,?,?,?,?)', rx)

    # Lab Results
    labs = [
        (1,  1, 1, 1, 'CBC',                '13.5/9500/230000',     '12-17/4500-11000/150000-400000', 3, (today-timedelta(days=13)).strftime('%Y-%m-%d')),
        (2,  1, 1, 1, 'Lipid Panel',        'TC:240 LDL:160 HDL:35','TC<200 LDL<130 HDL>40',          3, (today-timedelta(days=13)).strftime('%Y-%m-%d')),
        (3,  3, 5, 2, 'ECG',                'ST depression V4-V6',  'Normal sinus rhythm',            3, (today-timedelta(days=9)).strftime('%Y-%m-%d')),
        (4,  5, 4, 3, 'TSH',                '8.2 mIU/L',            '0.4-4.0 mIU/L',                  3, (today-timedelta(days=6)).strftime('%Y-%m-%d')),
        (5,  5, 4, 3, 'Free T4',            '0.7 ng/dL',            '0.8-1.8 ng/dL',                  3, (today-timedelta(days=6)).strftime('%Y-%m-%d')),
        (6,  7, 5, 4, 'Spirometry',         'FEV1/FVC = 0.58',      '>0.70',                          2, (today-timedelta(days=2)).strftime('%Y-%m-%d')),
        (7,  7, 5, 4, 'CXR',                'Hyperinflation noted', 'Clear lung fields',              2, (today-timedelta(days=2)).strftime('%Y-%m-%d')),
        (8,  2, 2, 2, 'Troponin I',         'Pending',              '<0.04 ng/mL',                    1, today.strftime('%Y-%m-%d')),
        (9,  9, 1, None,'Fasting Glucose',  'Pending',              '70-100 mg/dL',                   1, today.strftime('%Y-%m-%d')),
        (10, 8, 3, 5, 'Peak Flow',          '78% predicted',        '>80% predicted',                 3, (today-timedelta(days=1)).strftime('%Y-%m-%d')),
    ]
    for l in labs:
        c.execute('INSERT OR IGNORE INTO LabResults VALUES(?,?,?,?,?,?,?,?,?)', l)

    # Invoices
    invs = [
        (1,  1, 5,  350.0, 350.0, 2, (today-timedelta(days=7)).strftime('%Y-%m-%d')),
        (2,  3, 7,  600.0, 300.0, 3, (today-timedelta(days=3)).strftime('%Y-%m-%d')),
        (3,  5, None,200.0, 0.0,  1, (today-timedelta(days=7)).strftime('%Y-%m-%d')),
        (4,  7, None,450.0, 450.0,2, (today-timedelta(days=3)).strftime('%Y-%m-%d')),
        (5,  8, 8,  280.0, 0.0,  1, (today-timedelta(days=2)).strftime('%Y-%m-%d')),
        (6,  2, 2,  500.0, 0.0,  1, today.strftime('%Y-%m-%d')),
        (7,  6, 6,  150.0, 150.0,2, (today-timedelta(days=5)).strftime('%Y-%m-%d')),
        (8,  9, 9,  200.0, 0.0,  1, today.strftime('%Y-%m-%d')),
    ]
    for inv in invs:
        c.execute('INSERT OR IGNORE INTO Invoices VALUES(?,?,?,?,?,?,?)', inv)

    conn.commit()
    conn.close()
    print('✅ ClinicIQ database initialized successfully!')


def run_sql(query: str):
    """Execute SQL and return rows as list of dicts."""
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.row_factory = sqlite3.Row
        c = conn.cursor()
        c.execute(query)
        rows = [dict(r) for r in c.fetchall()]
        conn.close()
        return rows, None
    except Exception as e:
        return [], str(e)


VIEWS_SCHEMA = """
You have access to ONLY these 5 views — do NOT query raw tables directly:

1. AppointmentFull — All appointments with full context:
   Columns: AppointmentId, PatientName, PatientPhone, BloodType, KnownAllergies,
            ChronicDiseases, DoctorName, DoctorSpecialty, ClinicName,
            AppointmentDate (YYYY-MM-DD), AppointmentTime, Reason,
            Status (Scheduled/Completed/Cancelled/NoShow), Notes

2. MedicalRecordFull — Visit records with diagnoses and treatment:
   Columns: RecordId, PatientName, BloodType, KnownAllergies, ChronicDiseases,
            DoctorName, DoctorSpecialty, VisitDate, Diagnosis, Symptoms,
            TreatmentPlan, Prescriptions, LabTests, FollowUpDate, Notes

3. PatientSummary — Patient profile with visit stats:
   Columns: PatientId, FullName, DateOfBirth, Gender, Phone, NationalId,
            BloodType, KnownAllergies, ChronicDiseases, InsuranceNo,
            TotalAppointments, TotalVisits, LastAppointmentDate

4. LabResultsFull — Lab test results:
   Columns: LabId, PatientName, DoctorName, TestName, Result,
            ReferenceRange, LabStatus (Pending/Ready/Reviewed), TestedAt

5. InvoiceView — Billing and payment info:
   Columns: InvoiceId, PatientName, Amount, PaidAmount, BalanceDue,
            PaymentStatus (Pending/Paid/PartiallyPaid/Waived), IssuedAt
"""


def sql_agent(question: str, user_role: int = 1, user_name: str | None = None) -> tuple:
    """Translate NL question to SQL on views (RBAC-aware). Returns (rows, sql, err)."""
    client = _get_groq()
    role_label = {1: 'Receptionist', 2: 'Doctor', 3: 'Admin'}.get(user_role, 'Receptionist')

    if user_role == 1:
        rbac_note = (
            'This user is a Receptionist. They can see appointments and patient contact info '
            'but NOT full medical records, diagnoses, or lab details.'
        )
    elif user_role == 2 and user_name:
        rbac_note = (
            f"This user is a Doctor named '{user_name}'. "
            f"They can see their own patients' full records (filter by DoctorName LIKE '%{user_name.split()[-1]}%'). "
            f"They cannot see other doctors' records."
        )
    elif user_role == 2:
        rbac_note = 'This user is a Doctor. They can see their own patients full medical records.'
    else:
        rbac_note = 'This user is an Admin. They can see all data across all views.'

    prompt = f"""You are a SQL expert for ClinicIQ — a medical clinic management system.
Today's date: {datetime.now().strftime('%Y-%m-%d')}
Current user role: {role_label}
{rbac_note}

{VIEWS_SCHEMA}

Rules:
1. ONLY use SELECT queries — never INSERT, UPDATE, DELETE, DROP.
2. ONLY query from the 5 views listed above.
3. For name searches, always use LIKE '%name%' (case-insensitive).
4. For upcoming appointments: WHERE AppointmentDate >= date('now') AND Status = 'Scheduled'.
5. For today's appointments: WHERE AppointmentDate = date('now').
6. Receptionist role: NEVER query MedicalRecordFull or LabResultsFull.
7. Return ONLY the raw SQL query — no explanation, no markdown, no backticks.

Question: {question}
"""

    resp = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    sql = resp.choices[0].message.content.strip()
    sql = sql.replace('```sql', '').replace('```', '').strip()
    rows, err = run_sql(sql)
    return rows, sql, err


def format_sql_answer(question: str, rows: list, sql: str, user_role: int = 1) -> str:
    """Turn SQL rows into a natural-language answer (matches question language)."""
    if not rows:
        return 'No data found for your query.'
    client = _get_groq()
    prompt = f"""You are ClinicIQ AI — a medical clinic assistant.
Question: {question}
SQL results ({len(rows)} rows): {json.dumps(rows[:15], indent=2, default=str)}

Instructions:
- Answer in the EXACT same language as the question (Arabic/English/Mixed).
- Be concise and professional.
- Format dates as DD-MM-YYYY.
- Use bullet points for lists.
- NEVER invent data not in the results.
- For sensitive medical info, maintain professional tone.
"""
    resp = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return resp.choices[0].message.content


def evaluator_agent(
    stage: str,
    input_data,
    output_data,
    context: str = '',
    verbose: bool = False,
) -> dict:
    """
    QA evaluator after each pipeline stage.
    Returns dict with keys: score (0-10), verdict (pass|warn|fail), feedback.
    """
    client = _get_groq()
    prompt = f"""You are a strict QA evaluator for a medical AI system.

Stage: {stage}
Input: {str(input_data)[:400]}
Output: {str(output_data)[:600]}
Context: {str(context)[:200]}

Evaluate:
- Correctness: Is the output accurate for this stage?
- Completeness: Does it fully address the input?
- Format: Is it in the right format?

Respond ONLY in this JSON format:
{{"score": <0-10>, "verdict": "pass|warn|fail", "feedback": "<one sentence>"}}
"""
    resp = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    text = resp.choices[0].message.content.strip()
    try:
        result = json.loads(text[text.find('{') : text.rfind('}') + 1])
    except Exception:
        result = {'score': 5, 'verdict': 'warn', 'feedback': text[:100]}
    if verbose:
        icon = {'pass': '✅', 'warn': '⚠️', 'fail': '❌'}.get(result.get('verdict', ''), '❓')
        print(
            f"  {icon} Evaluator [{stage}] — {result.get('score', '?')}/10 | "
            f"{str(result.get('verdict', '')).upper()} | {result.get('feedback', '')}"
        )
    return result


def format_evaluations_summary(evals: list[dict]) -> str:
    if not evals:
        return ''
    parts = []
    for ev in evals:
        v  = ev.get('verdict', '?')
        s  = ev.get('score', '?')
        fb = (ev.get('feedback') or '')[:120]
        parts.append(f'{v} {s}/10 — {fb}')
    return 'Evaluator | ' + ' · '.join(parts)


def ask_sql(
    question: str,
    user_role: int = 3,
    user_name: str | None = None,
    verbose: bool = False,
    use_evaluator: bool = True,
) -> tuple[str, str | None, str | None, list[dict]]:
    rows, sql, err = sql_agent(question, user_role, user_name)
    evaluations: list[dict] = []

    if err:
        if use_evaluator:
            evaluations.append(
                evaluator_agent('sql_generation', question, sql, err, verbose=verbose)
            )
        if verbose:
            print(f'SQL error: {err}\nSQL was: {sql}')
        return f'Sorry, I could not query the database: {err}', sql, err, evaluations

    if use_evaluator:
        evaluations.append(
            evaluator_agent(
                'sql_generation', question, {'sql': sql, 'rows': rows[:3]}, '', verbose=verbose,
            )
        )

    answer = format_sql_answer(question, rows, sql or '', user_role)

    if use_evaluator:
        evaluations.append(
            evaluator_agent(
                'answer_quality', question, answer, json.dumps(rows[:3], default=str), verbose=verbose,
            )
        )

    if verbose:
        print(f'{len(rows)} rows | SQL: {sql}')

    return answer, sql, None, evaluations


# Initialize database on import
init_db()

✅ ClinicIQ database initialized successfully!


---
## 📚 Section 4 — RAG Layer

In [ ]:
# =============================================================================
# RAG layer
# =============================================================================

SUPPORTED_EXTENSIONS = {
    '.pdf', '.docx', '.doc', '.xlsx', '.xls', '.csv', '.txt',
    '.png', '.jpg', '.jpeg', '.webp',
}


def validate_file(path: str):
    if not os.path.exists(path):
        return False, f'File not found: {path}'
    if os.path.getsize(path) == 0:
        return False, f'File is empty: {path}'
    ext = os.path.splitext(path)[1].lower()
    if ext not in SUPPORTED_EXTENSIONS:
        return False, f"Unsupported type '{ext}'"
    return True, 'ok'


def validate_url(url: str):
    if not url.startswith(('http://', 'https://')):
        return False, f'Invalid URL: {url}'
    return True, 'ok'


def validate_knowledge_base(coll, emb: SentenceTransformer):
    errors = []
    if coll.count() == 0:
        errors.append('Knowledge base is empty')
        return False, errors
    try:
        e = emb.encode(['test']).tolist()
        results = coll.query(query_embeddings=e, n_results=1)
        if not results['documents'][0]:
            errors.append('Retrieval returned no results')
    except Exception as e:
        errors.append(f'Retrieval error: {e}')
    sample = coll.get(limit=5)['metadatas']
    for meta in sample:
        for key in ['source', 'page', 'type']:
            if key not in meta:
                errors.append(f"Missing metadata key '{key}'")
                break
    if errors:
        return False, errors
    all_m = coll.get()['metadatas']
    types   = Counter(m['type'] for m in all_m)
    sources = Counter(m['source'] for m in all_m)
    print(f'  Validation passed! {coll.count()} chunks | Types: {dict(types)}')
    for src, cnt in sources.most_common(5):
        print(f'    {src}: {cnt} chunks')
    return True, []


def extract_image_ocr(pil_img: Image.Image) -> str:
    """OCR — Arabic + English."""
    try:
        text = pytesseract.image_to_string(pil_img, lang='ara+eng')
        return text.strip()
    except Exception:
        return pytesseract.image_to_string(pil_img).strip()


def extract_pdf(path: str) -> list[dict]:
    chunks = []
    filename = os.path.basename(path)
    doc = fitz.open(path)
    for page_num, page in enumerate(doc, 1):
        text = page.get_text().strip()
        if text:
            chunks.append({'text': text, 'source': filename, 'page': page_num, 'type': 'text'})
        for img in page.get_images():
            try:
                img_bytes = doc.extract_image(img[0])['image']
                pil_img   = Image.open(io.BytesIO(img_bytes))
                ocr_text  = extract_image_ocr(pil_img)
                if ocr_text:
                    chunks.append({'text': f'[Image on page {page_num}]: {ocr_text}',
                                   'source': filename, 'page': page_num, 'type': 'image'})
            except Exception:
                pass
    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            for table in page.extract_tables() or []:
                if not table or not table[0]:
                    continue
                header   = ' | '.join(str(c or '') for c in table[0])
                sep      = ' | '.join(['---'] * len(table[0]))
                rows_str = ['\n'.join([header, sep]) + '\n' + '\n'.join(
                    ' | '.join(str(c or '') for c in row) for row in table[1:])]
                chunks.append({'text': f'[Table on page {page_num}]:\n{rows_str[0]}',
                               'source': filename, 'page': page_num, 'type': 'table'})
    return chunks


def extract_docx(path: str) -> list[dict]:
    doc  = DocxDoc(path)
    text = '\n'.join(p.text for p in doc.paragraphs if p.text.strip())
    return [{'text': text, 'source': os.path.basename(path), 'page': 1, 'type': 'text'}]


def extract_excel(path: str) -> list[dict]:
    dfs   = pd.read_excel(path, sheet_name=None)
    parts = [f'Sheet: {s}\n{df.to_string(index=False)}' for s, df in dfs.items()]
    return [{'text': '\n\n'.join(parts), 'source': os.path.basename(path), 'page': 1, 'type': 'text'}]


def extract_csv(path: str) -> list[dict]:
    text = pd.read_csv(path).to_string(index=False)
    return [{'text': text, 'source': os.path.basename(path), 'page': 1, 'type': 'text'}]


def extract_txt(path: str) -> list[dict]:
    with open(path, encoding='utf-8', errors='ignore') as f:
        text = f.read()
    return [{'text': text, 'source': os.path.basename(path), 'page': 1, 'type': 'text'}]


def extract_image_file(path: str) -> list[dict]:
    ocr_text = extract_image_ocr(Image.open(path))
    return [{'text': ocr_text or f'[Image: {os.path.basename(path)}]',
             'source': os.path.basename(path), 'page': 1, 'type': 'image'}]


EXTRACTORS = {
    '.pdf':  extract_pdf,
    '.docx': extract_docx, '.doc': extract_docx,
    '.xlsx': extract_excel, '.xls': extract_excel,
    '.csv':  extract_csv,
    '.txt':  extract_txt,
    '.png':  extract_image_file, '.jpg': extract_image_file,
    '.jpeg': extract_image_file, '.webp': extract_image_file,
}


def extract_file(path: str) -> list[dict]:
    ok, msg = validate_file(path)
    if not ok:
        print(f'  Skipping: {msg}')
        return []
    return EXTRACTORS[os.path.splitext(path)[1].lower()](path)


def file_hash(path: str) -> str:
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()


def semantic_split(items: list[dict]) -> list[dict]:
    """Semantic chunking; keep tables/images as small unsplit chunks."""
    result       = []
    lc_embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
    chunker = SemanticChunker(
        lc_embeddings,
        breakpoint_threshold_type='percentile',
        breakpoint_threshold_amount=85,
    )
    for item in items:
        if item['type'] in ('table', 'image') or len(item['text'].split()) < 50:
            result.append(item)
            continue
        try:
            docs = chunker.create_documents([item['text']])
            for doc in docs:
                result.append({'text': doc.page_content, 'source': item['source'],
                                'page': item['page'], 'type': item['type']})
        except Exception:
            result.append(item)
    return result


async def scrape(url: str, label: str, wait: int = 2000) -> str:
    ok, msg = validate_url(url)
    if not ok:
        print(f'  {msg}')
        return ''
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch()
            page    = await browser.new_page()
            await page.goto(url, timeout=30000, wait_until='domcontentloaded')
            await page.wait_for_timeout(wait)
            text = await page.inner_text('body')
            await browser.close()
            lines = [ln for ln in text.splitlines() if len(ln.strip()) > 30]
            return '\n'.join(lines[:400])
    except Exception as e:
        print(f'  {label}: {e}')
        return ''


def scrape_sync(url: str, label: str) -> str:
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(scrape(url, label))
    finally:
        loop.close()


def build_knowledge_base(extra_text: str | None = None) -> int:
    """Extract files, optional web scrape, chunk, embed, store in Chroma."""
    global embedder, collection, WEBSITES

    all_items: list[dict] = []
    processed_hashes: set[str] = set()

    for filename, url in PDF_URLS.items():
        if not os.path.exists(filename):
            ok, msg = validate_url(url)
            if not ok:
                print(f'  Skipping: {msg}')
                continue
            print(f'  Downloading {filename}...')
            r = requests.get(url, timeout=60, headers={'User-Agent': 'Mozilla/5.0'})
            if r.status_code == 200:
                open(filename, 'wb').write(r.content)
                print(f'  Saved {filename}')
            else:
                print(f'  HTTP {r.status_code} for {filename}')

    patterns = ['*.pdf','*.docx','*.doc','*.xlsx','*.xls','*.csv','*.txt',
                '*.png','*.jpg','*.jpeg','*.webp']
    all_files = [f for p in patterns for f in glob.glob(p)]
    print(f'Found {len(all_files)} file(s)')
    for f in all_files:
        fh = file_hash(f)
        if fh in processed_hashes:
            print(f'  Duplicate content skipped: {os.path.basename(f)}')
            continue
        try:
            items = extract_file(f)
            all_items.extend(items)
            processed_hashes.add(fh)
        except Exception as e:
            print(f'  {f}: {e}')

    if WEBSITES:
        print(f'Scraping {len(WEBSITES)} site(s)...')
        for label, url in WEBSITES.items():
            text = scrape_sync(url, label)
            if len(text) > 200:
                all_items.append({'text': text, 'source': label, 'page': 1, 'type': 'web'})

    if extra_text and extra_text.strip():
        all_items.append({'text': extra_text, 'source': 'Manual input', 'page': 1, 'type': 'text'})

    if not all_items:
        print('No content to index.')
        return 0

    print(f'Semantic chunking {len(all_items)} items...')
    chunks = semantic_split(all_items)
    print(f'  -> {len(chunks)} chunks')

    print('Embedding...')
    embedder = SentenceTransformer(EMBED_MODEL)
    chroma   = chromadb.Client()
    try:
        chroma.delete_collection('cliniciq_rag')
    except Exception:
        pass
    collection = chroma.create_collection('cliniciq_rag')
    texts = [c['text'] for c in chunks]
    metas = [{'source': c['source'], 'page': str(c['page']), 'type': c['type']} for c in chunks]
    embs  = embedder.encode(texts, show_progress_bar=True).tolist()
    collection.add(
        ids=[str(i) for i in range(len(texts))],
        embeddings=embs,
        documents=texts,
        metadatas=metas,
    )
    ok, _ = validate_knowledge_base(collection, embedder)
    return collection.count() if ok else 0


def search(question: str, n: int = 10):
    if collection is None or embedder is None:
        return [], []
    emb = embedder.encode([question]).tolist()
    res = collection.query(query_embeddings=emb, n_results=n)
    return res['documents'][0], res['metadatas'][0]


def ask(question: str) -> tuple:
    docs, metas = search(question)
    if not docs:
        return 'No relevant documents found.', [], 'bad', 'empty KB'

    context = '\n\n---\n\n'.join(
        f"[{m.get('source','?')} p.{m.get('page','?')}]\n{d}"
        for d, m in zip(docs, metas)
    )
    client = _get_groq()
    r = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                'role': 'system',
                'content': (
                    'You are HakimBot AI — a medical knowledge assistant. '
                    'Answer ONLY from the provided context. '
                    'Respond in the EXACT same language as the question. '
                    'Arabic question → Arabic ONLY. English → English ONLY. '
                    'Mixed → mirror the exact mix. '
                    'NEVER use Vietnamese, Thai, Russian, or any other unrelated language. '
                    'If the answer is not in the context, say so clearly. '
                    'Cite your source briefly at the end.'
                ),
            },
            {
                'role': 'user',
                'content': f'Context:\n{context}\n\nQuestion: {question}',
            },
        ],
        temperature=0.2,
    )
    answer = (r.choices[0].message.content or '').strip()

    ev = evaluator_agent('rag_answer', question, answer, context[:300])
    verdict = 'good' if ev.get('score', 0) >= 6 else 'bad'

    sources = list({m.get('source', '?') for m in metas})
    return answer, sources, verdict, ev.get('feedback', '')


def process_and_build(files, urls_text: str, manual_text: str) -> str:
    global WEBSITES
    if files:
        for f in files:
            dest = os.path.basename(f.name if hasattr(f, 'name') else f)
            shutil.copy(f.name if hasattr(f, 'name') else f, dest)
    if urls_text and urls_text.strip():
        WEBSITES = {}
        for line in urls_text.strip().splitlines():
            url = line.strip()
            if url:
                WEBSITES[url] = url
    n = build_knowledge_base(extra_text=manual_text or None)
    return f'✅ Knowledge base built: {n} chunks indexed.' if n else '⚠️ No content indexed.'

---
## 🌐 Section 5 — Web Search

In [ ]:
# =============================================================================
# Web search layer (DuckDuckGo)
# =============================================================================

def web_search(query: str, max_results: int = 5) -> str:
    """DuckDuckGo instant-answer scrape. Returns text snippets."""
    try:
        url = f'https://html.duckduckgo.com/html/?q={requests.utils.quote(query)}'
        r   = requests.get(url, timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        from html.parser import HTMLParser
        class _P(HTMLParser):
            def __init__(self):
                super().__init__()
                self.snips, self._c = [], False
            def handle_starttag(self, t, a):
                self._c = t == 'a' and any(k == 'class' and 'result__snippet' in v for k, v in a)
            def handle_data(self, d):
                if self._c and d.strip():
                    self.snips.append(d.strip()); self._c = False
        p = _P(); p.feed(r.text)
        return '\n'.join(p.snips[:max_results]) if p.snips else 'No results found.'
    except Exception as e:
        return f'Web search error: {e}'


def ask_web(question: str) -> str:
    snippets = web_search(question)
    if not snippets or snippets.startswith('Web search error'):
        return ''

    client = _get_groq()
    resp = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                'role': 'system',
                'content': (
                    'You are a helpful medical assistant. '
                    'Answer the question using ONLY the web search results provided. '
                    'Respond in the EXACT same language as the question. '
                    'Arabic question → Arabic ONLY, no other language. '
                    'English question → English ONLY. '
                    'Mixed → mirror the exact same mix. '
                    'NEVER use Vietnamese, Thai, Russian, or any other language. '
                    'Write clean flowing text. '
                    'Cite sources briefly at the end if helpful. '
                    'If the results do not answer the question, say so in one sentence.'
                ),
            },
            {'role': 'user', 'content': f'Web search results:\n{snippets}\n\nQuestion: {question}'},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content or ''

---
## 🔀 Section 6 — Smart Router + Memory + LangGraph

**Flow:**
- **Understand Layer** → intent + complexity + freshness
- **Decision Gate** → chitchat/vague → General LLM immediately
- **Planner** → decides sources (sql / rag / web)
- **Retrieval** → sequential (simple/medium) or **parallel** (complex)
- **Verification** → checks result quality
- **Smart Retry** → tries untried sources before giving up (max 2 retries)
- **Smart Merge** → single source returns directly; multi-source LLM synthesis
- **General LLM Fallback** → last resort


In [ ]:
from __future__ import annotations

import asyncio
import json as _json
import re
from typing import Annotated, Literal, Sequence, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages


# =============================================================================
# Memory
# =============================================================================

MEMORY_K = 6
_conv_memory: list[dict] = []
_conv_summary: str = ''


def _update_memory(q: str, a: str, route: str) -> None:
    global _conv_memory, _conv_summary
    _conv_memory.append({'q': q, 'a': a, 'route': route})
    if len(_conv_memory) > MEMORY_K:
        to_compress = _conv_memory[:-MEMORY_K]
        _conv_memory = _conv_memory[-MEMORY_K:]
        client = _get_groq()
        old_block = '\n'.join(
            f"User: {m['q']}\nAssistant: {m['a']}" for m in to_compress
        )
        prev = f"Previous summary:\n{_conv_summary}\n\n" if _conv_summary else ''
        r = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{
                'role': 'user',
                'content': (
                    f"{prev}Summarize the following conversation in 2-3 sentences, "
                    "keeping key medical facts and decisions. Same language as conversation.\n\n"
                    'Mixed → mirror the exact same mix. '
                    'NEVER use Vietnamese, Thai, Russian, or any other language. '
                    'Write clean flowing text — no \\n escape characters, use natural paragraph breaks only. '
                    f'{old_block}'
                ),
            }],
            temperature=0,
            max_tokens=200,
        )
        _conv_summary = r.choices[0].message.content.strip()


def get_memory_context() -> str:
    parts = []
    if _conv_summary:
        parts.append(f'[Conversation summary so far]:\n{_conv_summary}')
    for m in _conv_memory:
        parts.append(f"User: {m['q']}\nAssistant: {m['a']}")
    return '\n\n'.join(parts)


def clear_memory() -> None:
    global _conv_memory, _conv_summary
    _conv_memory = []
    _conv_summary = ''


# =============================================================================
# RBAC context
# =============================================================================

_runtime_user_role: int = 3
_runtime_user_name: str | None = None


def set_rbac_context(role: int, name: str | None) -> None:
    global _runtime_user_role, _runtime_user_name
    _runtime_user_role = int(role)
    _runtime_user_name = name.strip() if name else None


# =============================================================================
# Graph State
# =============================================================================

class GraphState(TypedDict):
    messages:     Annotated[Sequence[BaseMessage], add_messages]
    intent:       str
    complexity:   str
    freshness:    bool
    sources:      list
    rag_result:   str
    sql_result:   str
    web_result:   str
    retry_count:  int
    final:        str
    meta:         str


def _last_human(state: GraphState) -> str:
    for m in reversed(state['messages']):
        if isinstance(m, HumanMessage):
            return m.content if isinstance(m.content, str) else str(m.content)
    return ''


# =============================================================================
# Node 1 — Understand Layer
# =============================================================================

_UNDERSTAND_PROMPT = """You are a query analyst for ClinicIQ AI (medical clinic management + document Q&A).

Analyze the user message and return a JSON object with EXACTLY these three fields:

{
  "intent":     "chitchat" | "vague" | "real_question",
  "complexity": "simple"   | "medium" | "complex",
  "freshness":  true | false
}

Definitions:
- intent:
    "chitchat"      → greetings, thanks, small-talk
    "vague"         → under-specified, missing key context
    "real_question" → anything with a clear, answerable medical/clinic intent

- complexity:
    "simple"   → one clear fact, one source is enough
    "medium"   → needs two sources or moderate reasoning
    "complex"  → multi-part, cross-source, or requires synthesis of 3+ sources

- freshness:
    true  → involves current events, latest drug news, new guidelines, real-time data
    false → otherwise

Return ONLY the JSON object. No markdown, no explanation.
Handle English and Arabic messages correctly.
Respond in the EXACT same language as the question.
Arabic question → Arabic ONLY. English → English ONLY.
Mixed → mirror the exact same mix.
NEVER use Vietnamese, Thai, Russian, or any other unrelated language.
Write clean flowing text — no \n escape characters.
"""


def node_understand(state: GraphState) -> dict:
    text   = _last_human(state)
    client = _get_groq()
    try:
        r = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {'role': 'system', 'content': _UNDERSTAND_PROMPT},
                {'role': 'user',   'content': text.strip()},
            ],
            temperature=0,
            max_tokens=80,
        )
        raw = re.sub(r'```[a-z]*', '', r.choices[0].message.content or '').strip().strip('`').strip()
        parsed     = _json.loads(raw)
        intent     = parsed.get('intent',     'real_question')
        complexity = parsed.get('complexity', 'simple')
        freshness  = bool(parsed.get('freshness', False))
    except Exception:
        intent, complexity, freshness = 'real_question', 'simple', False

    t = text.lower()
    if re.search(r'\b(hi|hello|hey|thanks|مرحبا|شكراً|شكر|أهلا|هاي|أهلاً)\b', t):
        intent = 'chitchat'
    if re.search(r'\b(today|now|latest|current|recent|news|guidelines|دلوقتي|اليوم|الآن|أحدث|أخبار|النهارده)\b', t):
        freshness = True

    return {
        'intent':     intent,
        'complexity': complexity,
        'freshness':  freshness,
        'meta':       f'intent={intent}|complexity={complexity}|freshness={freshness}',
    }


def edge_after_understand(state: GraphState) -> str:
    intent = state.get('intent', 'real_question')
    if intent == 'chitchat': return 'general_chitchat'
    if intent == 'vague':    return 'general_clarify'
    return 'planner'


# =============================================================================
# Nodes 2a/2b/2c — General LLM
# =============================================================================

_BASE_LLM_SYSTEM = """Respond in the EXACT same language as the question.
Arabic question → Arabic ONLY. English → English ONLY.
Mixed → mirror the exact mix.
Respond in the EXACT same language as the question.
NEVER use Vietnamese, Thai, Russian, or any other language.
Write clean flowing text — no \n escape characters.
Skip garbled or unreadable text silently.
"""


def _general_llm_call(system_extra: str, text: str, temperature: float = 0.4) -> str:
    client = _get_groq()
    ctx    = get_memory_context()
    system = _BASE_LLM_SYSTEM + '\n\n' + system_extra
    if ctx:
        system += f'\n\nConversation context:\n{ctx}'
    r = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': text},
        ],
        temperature=temperature,
    )
    return (r.choices[0].message.content or '').strip()


def node_general_chitchat(state: GraphState) -> dict:
    text = _last_human(state)
    ans  = _general_llm_call(
        'You are HakimBot AI — a friendly medical clinic assistant. Reply warmly and briefly. No database or document lookup.',
        text,
    )
    _update_memory(text, ans, 'general_chitchat')
    return {'messages': [AIMessage(content=ans)], 'final': ans, 'meta': 'route=general_chitchat'}


def node_general_clarify(state: GraphState) -> dict:
    text = _last_human(state)
    ans  = _general_llm_call(
        ("You are HakimBot AI. The user's question is under-specified. "
         'Ask ONE concise clarifying question to understand what they need. '
         'Do not attempt to answer yet.'),
        text, temperature=0.3,
    )
    _update_memory(text, ans, 'general_clarify')
    return {'messages': [AIMessage(content=ans)], 'final': ans, 'meta': 'route=general_clarify'}


def node_general_fallback(state: GraphState) -> dict:
    text = _last_human(state)
    ans  = _general_llm_call(
        ('You are HakimBot AI. No database or document results were found. '
         'Answer from your medical knowledge as best you can. '
          'Respond in the EXACT same language as the question. '
          'Arabic question → Arabic ONLY, no other language. '
          'English question → English ONLY. '
          'Mixed → mirror the exact same mix. '
          'NEVER use Vietnamese, Thai, Russian, or any other language. '
          'Write clean flowing text — no \\n escape characters, use natural paragraph breaks only. '
         'If you are not confident, say so briefly.'),
        text, temperature=0.4,
    )
    _update_memory(text, ans, 'general_fallback')
    badge   = '🧠 llm-fallback'
    content = f'{ans}\n\n{badge}'
    return {'messages': [AIMessage(content=content)], 'final': content, 'meta': 'route=general_fallback'}


# =============================================================================
# Node 3 — Planner
# =============================================================================

_PLANNER_PROMPT = """You are a retrieval planner for HakimBot AI (medical clinic system).

Given a user question, decide which data sources are needed.
Available sources: sql, rag, web

- sql  → patient data, appointments, medical records, lab results, prescriptions, billing — structured DB data
- rag  → medical books, clinical guidelines, reports, uploaded documents
- web  → latest medical news, new drug approvals, current treatment guidelines

Return a JSON array with the sources to use, ordered by priority.
Examples:
  "what appointments does Dr. Ahmed have today?"  → ["sql"]
  "what does the hypertension protocol say?"      → ["rag"]
  "latest COVID treatment guidelines"             → ["web"]
  "compare patient's labs vs normal range"        → ["sql", "rag"]
  "Mohamed Ali's diagnosis + treatment options"   → ["sql", "rag"]
  "complex multi-source medical question"         → ["sql", "rag", "web"]

Return ONLY the JSON array. No markdown, no explanation."""


def node_planner(state: GraphState) -> dict:
    text       = _last_human(state)
    complexity = state.get('complexity', 'simple')
    freshness  = state.get('freshness',  False)
    client     = _get_groq()

    try:
        r = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {'role': 'system', 'content': _PLANNER_PROMPT},
                {'role': 'user',   'content': text.strip()},
            ],
            temperature=0, max_tokens=40,
        )
        raw     = re.sub(r'```[a-z]*', '', r.choices[0].message.content or '').strip()
        sources = _json.loads(raw)
        sources = [s for s in sources if s in ('sql', 'rag', 'web')]
    except Exception:
        sources = []

    if not sources:
        t = text.lower()
        if re.search(r'\b(patient|appointment|doctor|record|prescription|lab|invoice|clinic|مريض|موعد|دكتور|تقرير|دواء|تحليل)\b', t):
            sources = ['sql']
        elif freshness:
            sources = ['web']
        else:
            sources = ['rag']

    if freshness and 'web' not in sources and complexity == 'complex':
        sources.append('web')

    if complexity == 'complex' and len(sources) < 2:
        sources = (['sql'] + sources) if 'sql' not in sources else (sources + ['rag'])

    return {
        'sources':     sources,
        'rag_result':  '',
        'sql_result':  '',
        'web_result':  '',
        'retry_count': 0,
        "meta":        f"sources={sources}",
    }


# =============================================================================
# Node 4 — Retrieval Layer
# =============================================================================

def _run_rag(text: str) -> str:
    try:
        ans, sources, verdict, note = ask(text)
        if verdict == 'bad' or not ans:
            return ''
        bad = ('not in the context', 'لم أجد', 'لا توجد', 'no information')
        if any(p in ans.lower() for p in bad):
            return ''
        return ans.strip()
    except Exception:
        return ''


def _run_sql(text: str) -> str:
    try:
        ans, _sql, err, evals = ask_sql(
            text,
            user_role=_runtime_user_role,
            user_name=_runtime_user_name,
            verbose=False,
            use_evaluator=True,
        )
        if err:
            return ''
        bad = ('no data found', 'could not query', 'no results')
        if any(p in (ans or '').lower() for p in bad):
            return ''
        footer = format_evaluations_summary(evals)
        return (ans + (f'\n\n_{footer}_' if footer else '')).strip()
    except Exception:
        return ''


def _run_web(text: str) -> str:
    try:
        return (ask_web(text) or '').strip()
    except Exception:
        return ''


def node_retrieval(state: GraphState) -> dict:
    text       = _last_human(state)
    sources    = state.get('sources', ['rag'])
    complexity = state.get('complexity', 'simple')

    rag_result = state.get('rag_result', '')
    sql_result = state.get('sql_result', '')
    web_result = state.get('web_result', '')

    if complexity == 'complex' and len(sources) > 1:
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            futures = {}
            if 'rag' in sources: futures['rag'] = pool.submit(_run_rag, text)
            if 'sql' in sources: futures['sql'] = pool.submit(_run_sql, text)
            if 'web' in sources: futures['web'] = pool.submit(_run_web, text)
            results = {k: f.result() for k, f in futures.items()}
        rag_result = results.get('rag', '')
        sql_result = results.get('sql', '')
        web_result = results.get('web', '')
    else:
        for src in sources:
            if src == 'rag' and not rag_result: rag_result = _run_rag(text)
            elif src == 'sql' and not sql_result: sql_result = _run_sql(text)
            elif src == 'web' and not web_result: web_result = _run_web(text)

    return {'rag_result': rag_result, 'sql_result': sql_result, 'web_result': web_result}


# =============================================================================
# Node 5 — Verification Layer
# =============================================================================

MAX_RETRIES = 2


def node_verify(state: GraphState) -> dict:
    return {}


def edge_after_verify(state: GraphState) -> str:
    has_data = bool(
        (state.get('rag_result') or '').strip() or
        (state.get('sql_result') or '').strip() or
        (state.get('web_result') or '').strip()
    )
    if has_data:
        return 'merge_and_answer'
    if state.get('retry_count', 0) < MAX_RETRIES:
        return 'smart_retry'
    return 'general_fallback'


# =============================================================================
# Node 6 — Smart Retry
# =============================================================================

def node_smart_retry(state: GraphState) -> dict:
    tried   = state.get('sources', [])
    untried = [s for s in ['sql', 'rag', 'web'] if s not in tried]
    new_src = tried + (untried[:1] if untried else ['web'])
    return {
        'sources':     new_src,
        'rag_result':  '',
        'sql_result':  '',
        'web_result':  '',
        'retry_count': state.get('retry_count', 0) + 1,
        'meta':        f"retry#{state.get('retry_count',0)+1}|sources={new_src}",
    }


# =============================================================================
# Node 7 — Smart Merge + Answer Generator
# =============================================================================

def node_merge_and_answer(state: GraphState) -> dict:
    text = _last_human(state)
    sql  = (state.get('sql_result') or '').strip()
    rag  = (state.get('rag_result') or '').strip()
    web  = (state.get('web_result') or '').strip()

    filled = [(lbl, cnt) for lbl, cnt in
              [('Database', sql), ('Documents', rag), ('Web', web)] if cnt]

    if not filled:
        return {'final': '', 'meta': 'merge=empty'}

    if len(filled) == 1:
        label, content = filled[0]
        badge = {'Database': '🗄️ sql', 'Documents': '📚 rag', 'Web': '🌐 web'}.get(label, '')
        final = f'{content}\n\n{badge}'
        _update_memory(text, content, label.lower())
        return {'messages': [AIMessage(content=final)], 'final': final,
                'meta': f'merge=single|source={label}'}

    client = _get_groq()
    ctx    = get_memory_context()
    sources_block = '\n\n'.join(f'--- {lbl} ---\n{cnt}' for lbl, cnt in filled)

    r = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                'role': 'system',
                'content': (
                    'You are HakimBot AI. Merge the provided source results into ONE '
                    "coherent medical answer in the user's language. "
                    'Respond in the EXACT same language as the question. '
                    'Arabic question → Arabic ONLY. English → English ONLY. '
                    'Mixed → mirror the exact same mix. '
                    'NEVER use Vietnamese, Thai, Russian, or any other language. '
                    "Write clean flowing text — no \\n escape characters. "
                    'Priority: Database > Documents > Web. '
                    'Use clear sections if multiple sources contributed meaningfully. '
                    "If a source is partial, integrate what's useful and skip the rest. "
                    + _BASE_LLM_SYSTEM
                    + (f'\n\nConversation context:\n{ctx}' if ctx else '')
                ),
            },
            {'role': 'user', 'content': f'Question:\n{text}\n\n{sources_block}'},
        ],
        temperature=0.2,
    )
    merged = (r.choices[0].message.content or '').strip()
    badge  = ' '.join(
        e for e in [
            '\U0001f5c4' if any(l == 'Database'  for l, _ in filled) else '',
            '\U0001f4da' if any(l == 'Documents' for l, _ in filled) else '',
            '\U0001f310' if any(l == 'Web'       for l, _ in filled) else '',
        ] if e
    )
    labels = '+'.join(l.lower()[:3] for l, _ in filled)
    final  = f'{merged}\n\n{badge}'
    _update_memory(text, merged, f'merge:{labels}')
    return {'messages': [AIMessage(content=final)], 'final': final,
            'meta': f'merge=multi|sources={labels}'}


# =============================================================================
# Build LangGraph
# =============================================================================

def build_router_graph():
    g = StateGraph(GraphState)

    g.add_node('understand',       node_understand)
    g.add_node('general_chitchat', node_general_chitchat)
    g.add_node('general_clarify',  node_general_clarify)
    g.add_node('general_fallback', node_general_fallback)
    g.add_node('planner',          node_planner)
    g.add_node('retrieval',        node_retrieval)
    g.add_node('verify',           node_verify)
    g.add_node('smart_retry',      node_smart_retry)
    g.add_node('merge_and_answer', node_merge_and_answer)

    g.add_edge(START, 'understand')

    g.add_conditional_edges(
        'understand',
        edge_after_understand,
        {
            'general_chitchat': 'general_chitchat',
            'general_clarify':  'general_clarify',
            'planner':          'planner',
        },
    )

    g.add_edge('general_chitchat', END)
    g.add_edge('general_clarify',  END)
    g.add_edge('planner',          'retrieval')
    g.add_edge('retrieval',        'verify')

    g.add_conditional_edges(
        'verify',
        edge_after_verify,
        {
            'merge_and_answer': 'merge_and_answer',
            'smart_retry':      'smart_retry',
            'general_fallback': 'general_fallback',
        },
    )

    g.add_edge('smart_retry',      'retrieval')
    g.add_edge('merge_and_answer', END)
    g.add_edge('general_fallback', END)

    return g.compile()


ROUTER_APP = build_router_graph()


def chat_router(message: str) -> tuple[str, str]:
    out = ROUTER_APP.invoke(
        {
            'messages':    [HumanMessage(content=message)],
            'intent':      '',
            'complexity':  '',
            'freshness':   False,
            'sources':     [],
            'rag_result':  '',
            'sql_result':  '',
            'web_result':  '',
            'retry_count': 0,
            'final':       '',
            'meta':        '',
        }
    )
    return out.get('final', ''), out.get('meta', '')

---
## 🖥️ Section 7 — Gradio UI

In [ ]:
# =============================================================================
# Gradio UI — ClinicIQ Medical Assistant
# =============================================================================

import gradio as gr

CSS = """
body, .gradio-container { background: #0a1628 !important; }

.gr-button-primary, button.primary {
    background: linear-gradient(135deg, #0a4a7a, #0d6eac, #0a4a7a) !important;
    border: 1px solid #1a8fd1 !important;
    color: #e0f4ff !important;
    font-weight: 600 !important;
    box-shadow: 0 0 20px rgba(26,143,209,0.3) !important;
    transition: all 0.2s ease !important;
}
.gr-button-primary:hover, button.primary:hover {
    box-shadow: 0 0 35px rgba(26,143,209,0.6) !important;
    border-color: #5bb8f5 !important;
    color: white !important;
}

.gr-button-secondary, button.secondary {
    background: #0d1f38 !important;
    border: 1px solid #1e3a5f !important;
    color: #7ec8e3 !important;
    transition: all 0.2s ease !important;
}
.gr-button-secondary:hover, button.secondary:hover {
    border-color: #1a8fd1 !important;
    box-shadow: 0 0 10px rgba(26,143,209,0.25) !important;
}

.gr-textbox textarea, .gr-textbox input {
    background: #0d1f38 !important;
    border: 1px solid #1e3a5f !important;
    color: #e2e8f0 !important;
    border-radius: 10px !important;
}
.gr-textbox textarea:focus, .gr-textbox input:focus {
    border-color: #1a8fd1 !important;
    box-shadow: 0 0 12px rgba(26,143,209,0.35) !important;
}

.gr-box, .gr-panel {
    background: #0d1f38 !important;
    border: 1px solid #1e3a5f !important;
    border-radius: 16px !important;
}

.gr-chatbot { background: #0a1628 !important; border: 1px solid #1e3a5f !important; border-radius: 16px !important; }
.gr-chatbot .message.user { background: #0d3060 !important; border-radius: 12px !important; }
.gr-chatbot .message.bot  { background: #0d1f38 !important; border-radius: 12px !important; color: #e2e8f0 !important; }

.sidebar-item {
    padding: 8px 12px; margin: 4px 0; border-radius: 8px;
    background: #0d1f38; cursor: pointer; border: 1px solid #1e3a5f;
    font-size: 13px; color: #7ec8e3;
}
.sidebar-item:hover { background: #1a3a5c; border-color: #1a8fd1; color: #e0f4ff; }
.sidebar-item.active { background: #0d3060; border-color: #1a8fd1; color: white; font-weight: 600; }
"""


_all_conversations: dict[str, dict] = {}
_active_conv_id: str = ''


def _new_conv_id() -> str:
    import uuid
    return str(uuid.uuid4())[:8]


def _save_active(messages, summary):
    global _active_conv_id
    if not _active_conv_id:
        _active_conv_id = _new_conv_id()
    if _active_conv_id not in _all_conversations:
        _all_conversations[_active_conv_id] = {'title': 'New Chat', 'messages': []}
    _all_conversations[_active_conv_id]['messages'] = messages
    if messages:
        first_user = next((m['content'] for m in messages if m.get('role') == 'user'), '')
        _all_conversations[_active_conv_id]['title'] = first_user[:45] + ('…' if len(first_user) > 45 else '')


def _history_html() -> str:
    if not _all_conversations:
        return "<p style='color:#5a8aa0;font-size:13px;'>No history yet.</p>"
    items = []
    for cid, cv in reversed(list(_all_conversations.items())):
        active = 'active' if cid == _active_conv_id else ''
        title  = cv.get('title', 'Chat')[:40]
        items.append(
            f"<div class='sidebar-item {active}' "
            f"onclick=\"document.getElementById('load_conv_id').querySelector('textarea').value='{cid}';"
            f"document.getElementById('load_conv_btn').click();\">"
            f"{title}</div>"
        )
    return ''.join(items)


def respond(message: str, history: list, role_str: str, user_name_str: str):
    role_map = {'Admin': 3, 'Doctor': 2, 'Receptionist': 1}
    role_int = role_map.get(role_str, 3)
    set_rbac_context(role_int, user_name_str or None)

    try:
        reply, meta = chat_router(message)
        if not reply:
            reply = '⚠️ No answer found. Please try rephrasing your question.'
        history = history + [
            {'role': 'user',      'content': message},
            {'role': 'assistant', 'content': reply},
        ]
    except Exception as e:
        history = history + [
            {'role': 'user',      'content': message},
            {'role': 'assistant', 'content': f'Error: {e}'},
        ]
    _save_active(history, _conv_summary)
    return '', history, _history_html()


def new_conv():
    global _active_conv_id
    _active_conv_id = _new_conv_id()
    clear_memory()
    return [], _history_html()


def load_conv(conv_id):
    global _active_conv_id
    cv = _all_conversations.get(conv_id.strip())
    if not cv:
        return gr.update(), _history_html()
    _active_conv_id = conv_id.strip()
    clear_memory()
    msgs = cv.get('messages', [])
    for i in range(0, len(msgs) - 1, 2):
        if msgs[i]['role'] == 'user' and i+1 < len(msgs):
            _update_memory(msgs[i]['content'], msgs[i+1]['content'], 'loaded')
    return cv.get('messages', []), _history_html()


def rename_conv(new_title):
    if _active_conv_id and _active_conv_id in _all_conversations:
        _all_conversations[_active_conv_id]['title'] = new_title[:60]
    return _history_html()


def delete_conv():
    global _active_conv_id
    if _active_conv_id in _all_conversations:
        del _all_conversations[_active_conv_id]
    _active_conv_id = ''
    clear_memory()
    return [], _history_html()


def launch_gradio(share: bool = False):
    with gr.Blocks(title='🏥 HakimBot Medical Assistant', css=CSS) as demo:

        gr.HTML("""
        <div style='text-align:center;padding:24px;
             background:linear-gradient(135deg,#061428,#0a2d52,#0d3a6b);
             border-radius:16px;margin-bottom:20px;
             border:1px solid #1a8fd1;
             box-shadow:0 0 40px rgba(26,143,209,0.3);'>
            <div style='font-size:44px;margin-bottom:8px;'>🏥</div>
            <h1 style='color:#7ec8e3;margin:0;font-size:26px;font-weight:700;letter-spacing:-0.3px;'>
                HakimBot
            </h1>
            <p style='color:#5bb8f5;margin:4px 0 0;font-size:15px;font-weight:500;'>
                Medical Intelligence Assistant
            </p>
            <p style='color:#4a8fa8;margin:8px 0 0;font-size:13px;'>
                Router: 💬 chitchat &nbsp;|&nbsp; ❓ clarify &nbsp;|&nbsp; 📚 rag &nbsp;|&nbsp; 🗄️ sql &nbsp;|&nbsp; 🌐 web &nbsp;|&nbsp; 🛟 fallback
            </p>
        </div>
        """)

        with gr.Row():
            # ── LEFT: History sidebar ─────────────────────────────────────────
            with gr.Column(scale=1, min_width=200):
                gr.Markdown('### 💬 Conversations')
                new_btn     = gr.Button('＋ New chat', variant='primary')
                history_box = gr.HTML(value="<p style='color:#4a8fa8;font-size:13px;'>No history yet.</p>")
                rename_txt  = gr.Textbox(label='Rename active chat', placeholder='New title…', lines=1)
                rename_btn  = gr.Button('✏️ Rename', variant='secondary')
                del_btn     = gr.Button('🗑️ Delete chat', variant='secondary')
                load_conv_id  = gr.Textbox(visible=False, elem_id='load_conv_id')
                load_conv_btn = gr.Button(visible=False, elem_id='load_conv_btn')

            # ── MIDDLE: RAG + Role settings ───────────────────────────────────
            with gr.Column(scale=1):
                gr.Markdown('### 📚 Medical Knowledge Base (RAG)')
                file_upload = gr.File(
                    label='Upload medical books, reports, guidelines…',
                    file_count='multiple',
                    file_types=[
                        '.pdf', '.docx', '.doc', '.xlsx', '.xls',
                        '.csv', '.txt', '.png', '.jpg', '.jpeg', '.webp',
                    ],
                )
                urls_input   = gr.Textbox(label='Medical websites (one URL per line)', lines=2)
                text_input   = gr.Textbox(label='Manual medical notes', lines=2)
                build_btn    = gr.Button('Build knowledge base', variant='primary')
                build_status = gr.Textbox(label='Status', interactive=False)
                build_btn.click(
                    process_and_build,
                    [file_upload, urls_input, text_input],
                    [build_status],
                )
                gr.Markdown('### 🔐 Access Role (RBAC)')
                role_dd = gr.Dropdown(
                    choices=['Admin', 'Doctor', 'Receptionist'],
                    value='Admin',
                    label='Select your role',
                )
                emp_name = gr.Textbox(
                    label='Doctor full name (for Doctor role — filters to your patients)',
                    placeholder='e.g. Dr. Ahmed Samir',
                )

            # ── RIGHT: Chat ──────────────────────────────────────────────────
            with gr.Column(scale=2):
                chatbot = gr.Chatbot(height=480, type='messages', allow_tags=False)
                msg  = gr.Textbox(placeholder='Ask about patients, appointments, diagnoses, medical books…')
                with gr.Row():
                    send = gr.Button('Send', variant='primary')
                    clr  = gr.Button('Clear chat', variant='secondary')

        history_state = gr.State([])

        def do_send(m, h, r, n):
            return respond(m, h, r, n)

        msg.submit(do_send, [msg, history_state, role_dd, emp_name],
                   [msg, history_state, history_box])
        send.click(do_send, [msg, history_state, role_dd, emp_name],
                   [msg, history_state, history_box])
        history_state.change(lambda h: h, inputs=[history_state], outputs=[chatbot])
        clr.click(lambda: ([], _history_html()), outputs=[history_state, history_box])

        new_btn.click(new_conv, outputs=[history_state, history_box])
        history_state.change(lambda h: h, inputs=[history_state], outputs=[chatbot])

        rename_btn.click(rename_conv, inputs=[rename_txt], outputs=[history_box])
        del_btn.click(delete_conv, outputs=[history_state, history_box])
        load_conv_btn.click(load_conv, inputs=[load_conv_id],
                            outputs=[history_state, history_box])
        history_state.change(lambda h: h, inputs=[history_state], outputs=[chatbot])

    demo.launch(share=share, show_error=True)


if __name__ == '__main__':
    launch_gradio(share=True)

/tmp/ipykernel_40227/2768368422.py:162: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title='🏥 HakimBot Medical Assistant', css=CSS) as demo:
/tmp/ipykernel_40227/2768368422.py:228: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=480, type='messages', allow_tags=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://748eb93cb37dd72b12.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
